In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mathurinache/flores101")

print("Path to dataset files:", path)

100%|██████████| 13.0M/13.0M [00:00<00:00, 143MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mathurinache/flores101/versions/1


In [ ]:
import os

# The dataset was downloaded to the 'path' variable from the previous cell.
# The typical structure of flores101 dataset is flores101/flores101_dataset/devtest
base_path = os.path.join(path, "flores101_dataset", "devtest")

def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f]

SRC = "eng"
TGT = "zul"

sources = load_lines(f"{base_path}/{SRC}.devtest")
references = load_lines(f"{base_path}/{TGT}.devtest")

print("Eval samples:", len(sources))

Eval samples: 1012


In [ ]:
from datasets import load_dataset

# Chosen corpus: OPUS100
# dataset = load_dataset("opus100", "en-zu")
dataset = load_dataset("Helsinki-NLP/opus-100", "en-zu")

train_data = dataset["train"].select(range(20000))  # small subset for Colab

def preprocess(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["zu"]
    }

train_data = train_data.map(preprocess)

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-zu/test-00000-of-00001.parquet:   0%|          | 0.00/58.1k [00:00<?, ?B/s]

en-zu/train-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

en-zu/validation-00000-of-00001.parquet:   0%|          | 0.00/58.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/38616 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "AfriNLP/AfriNLLB-12enc-12dec-full-ft-kd"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
def tokenize(example):
    model_inputs = tokenizer(
        example["src"],
        text_target=example["tgt"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    return model_inputs

tokenized_train = train_data.map(tokenize, batched=True)

print(tokenized_train[0].keys())

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

dict_keys(['translation', 'src', 'tgt', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./afrinllb-retrained",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    optim="adamw_torch"  # Explicitly used non-fused optimizer to avoid XLA compatibility issues
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

trainer.train()

Step,Training Loss
100,0.319729
200,0.221073
300,0.207167
400,0.206520
500,0.179381
600,0.174561
700,0.169262
800,0.155358
900,0.164211
1000,0.164103


Step,Training Loss
100,0.319729
200,0.221073
300,0.207167
400,0.206520
500,0.179381
600,0.174561
700,0.169262
800,0.155358
900,0.164211
1000,0.164103


TrainOutput(global_step=5000, training_loss=0.1331661642074585, metrics={'train_runtime': 2991.601, 'train_samples_per_second': 13.371, 'train_steps_per_second': 1.671, 'total_flos': 1.083552301056e+16, 'train_loss': 0.1331661642074585, 'epoch': 2.0})

In [ ]:
# Move model to CPU to resolve invalid storage pointer issues during serialization
model.to("cpu")
model.save_pretrained("afrinllb-retrained", safe_serialization=False)
tokenizer.save_pretrained("afrinllb-retrained")
# Move back to original device if needed for further processing
model.to(device)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
import time

def translate(texts, batch_size=8):
    tokenizer.src_lang = "eng_Latn"
    outputs = []

    start = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("zul_Latn"),
                max_length=200
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)

    total = time.time() - start
    latency = total / len(texts)
    throughput = len(texts) / total

    return outputs, latency, throughput

In [ ]:
predictions, latency, throughput = translate(sources)

print("Latency:", latency)
print("Throughput:", throughput)

Latency: 0.17662601061018088
Throughput: 5.66168027316787


In [ ]:
import sacrebleu

chrf = sacrebleu.corpus_chrf(predictions, [references])
print("chrF++:", chrf.score)

chrF++: 54.696999151987654


In [ ]:
import json

# =========================================================
# Save retrained model metrics
# =========================================================

results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf.score,
    "latency": latency,
    "throughput": throughput,
    "num_samples": len(sources)
}

# Save metrics
with open("retrained_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# =========================================================
# Save retrained model predictions
# =========================================================

with open("retrained_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

# =========================================================
# Optional but strongly recommended:
# Save evaluation data for separate AfriCOMET notebook
# =========================================================

with open("retrained_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("retrained_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)

print("Saved retrained evaluation files.")

Saved retrained evaluation files.


#### Refinement RAT/RAG

In [2]:
import unicodedata
import re

def clean_translation(text: str) -> str:
    """
    Applies Unicode normalization and regex heuristics to clean up
    common machine translation and subword tokenization artifacts.
    """
    if not isinstance(text, str):
        return ""

    # ---------------------------------------------------------
    # 1. Unicode Normalization
    # ---------------------------------------------------------
    # NFC (Normalization Form Canonical Composition) ensures that
    # characters are fully composed. This is critical for exact-match
    # metrics like BLEU and chrF++.
    text = unicodedata.normalize('NFC', text)

    # ---------------------------------------------------------
    # 2. Fix Punctuation Spacing (BPE Artifacts)
    # ---------------------------------------------------------
   # Remove spaces placed mistakenly before terminal punctuation
    text = re.sub(r'\s+([?.!,:;])', r'\1', text)

    # Fix spacing inside parentheses and brackets
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\[\s+', '[', text)
    text = re.sub(r'\s+\]', ']', text)

    # Fix basic quotation spacing (assumes standard English/Zulu quotes)
    text = re.sub(r'(["\'])\s+(.*?)\s+\1', r'\1\2\1', text)

    # ---------------------------------------------------------
    # 3. Clean up Whitespace
    # ---------------------------------------------------------
    # Collapse multiple spaces into a single space
    text = re.sub(r'\s{2,}', ' ', text)

    # Strip leading and trailing whitespace
    return text.strip()

# =========================================================
# Example Usage in your Evaluation Loop
# =========================================================


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("venkataanuhyatummala/flores200data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/venkataanuhyatummala/flores200data


In [ ]:
# Add this as a new cell in your notebook
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
# Replace your FAISS indexing cell with this:
print("Encoding reference library...")
reference_embeddings = retriever.encode(reference_sources, convert_to_numpy=True)

# Normalize for Cosine Similarity
faiss.normalize_L2(reference_embeddings)

dimension = reference_embeddings.shape[1]
# IndexFlatIP calculates Inner Product (which equals Cosine Similarity on normalized vectors)
index = faiss.IndexFlatIP(dimension)
index.add(reference_embeddings)

print(f"FAISS vector index built successfully with {index.ntotal} sentences.")

Encoding reference library...
FAISS vector index built successfully with 997 sentences.


In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList

class BagOfWordsLogitsProcessor(LogitsProcessor):
    def __init__(self, target_token_ids, bias_strength=1.5):
        # Use a set to prevent explosive boosting on repeated tokens
        # and remove special tokens if necessary
        self.target_token_ids = torch.tensor(list(set(target_token_ids)), dtype=torch.long, device=device)
        self.bias_strength = bias_strength

    def __call__(self, input_ids, scores):
        # Apply a soft boost to the allowed vocabulary independent of sequence
        scores[:, self.target_token_ids] += self.bias_strength
        return scores

In [ ]:
def translate_with_rat(texts, similarity_threshold=0.75, top_k=1, bias=2.0):
    outputs = []
    start = time.time()

    # Batch encode and normalize sources
    embeddings = retriever.encode(texts, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)

    # Search
    distances, indices = index.search(embeddings, top_k)

    for i, text in enumerate(texts):
        sim_score = distances[i][0]

        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        processor_list = []

        # Only apply constraints if the retrieved match is semantically close
        if sim_score >= similarity_threshold:
            retrieved_target = reference_targets[indices[i][0]]
            target_ids = tokenizer(retrieved_target, add_special_tokens=False).input_ids
            processor_list.append(BagOfWordsLogitsProcessor(target_ids, bias_strength=bias))

        processors = LogitsProcessorList(processor_list)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("zul_Latn"),
                max_length=200,
                num_beams=5,
                logits_processor=processors
            )

        decoded = clean_translation(tokenizer.decode(generated[0], skip_special_tokens=True))
        outputs.append(decoded)

    latency = (time.time() - start) / len(texts)
    throughput = len(texts) / (time.time() - start)
    return outputs, latency, throughput

In [ ]:
print("Running Retrieval-Augmented Translation...")
# Generate predictions using the RAT function
predictions_rat, latency_rat, throughput_rat = translate_with_rat(sources, top_k=1)

with open("rat_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions_rat, f, ensure_ascii=False, indent=2)
import sacrebleu

# Compare against the baseline metric (chrf.score) calculated earlier
chrf_rat = sacrebleu.corpus_chrf(predictions_rat, [references])

print("\n=== Evaluation Results ===")
print(f"Baseline chrF++:   {chrf.score:.2f}")
print(f"RAT chrF++:        {chrf_rat.score:.2f}")
print(f"RAT Latency:       {latency_rat:.4f} sec/sample")
print(f"RAT Throughput:    {throughput_rat:.2f} samples/sec")

Running Retrieval-Augmented Translation...

=== Evaluation Results ===
Baseline chrF++:   54.70
RAT chrF++:        56.29
RAT Latency:       0.8854 sec/sample
RAT Throughput:    1.13 samples/sec


In [ ]:
results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf_rat.score,
    "latency": latency_rat,
    "throughput": throughput_rat,
    "num_samples": len(sources)
}

# Save metrics, refs and sources
with open("rag_rat_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
with open("rag_rat_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("rag_rat_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)

In [ ]:
import torch
import time

def translate_with_lexical_constraints(texts, retriever, index, ref_targets, top_k=1, similarity_threshold=0.75):
    tokenizer.src_lang = "eng_Latn"
    outputs = []

    start = time.time()

    # 1. Batch encode all texts at once to maximize GPU throughput and decrease latency
    embeddings = retriever.encode(texts, convert_to_numpy=True)

    # If your FAISS index uses Cosine Similarity (IndexFlatIP with normalized vectors), uncomment line below:
    # faiss.normalize_L2(embeddings)

    distances, indices = index.search(embeddings, top_k)

    # 2. Iterate through sentences and apply conditional hard constraints
    for i, text in enumerate(texts):
        sim_score = distances[i][0]
        idx = indices[i][0]

        force_words = None

        # 3. Only enforce constraints if the retrieved template is high quality
        # Note: If your index is IndexFlatIP (Cosine), use: sim_score >= similarity_threshold
        # Note: If your index is default IndexFlatL2, use: sim_score <= similarity_threshold (e.g., threshold = 10.0)
        if sim_score >= similarity_threshold:
            retrieved_zulu = ref_targets[idx]

            # Strip out punctuation to prevent forcing formatting characters
            words = [w.strip(".,!?()-\"';:") for w in retrieved_zulu.split()]
            # Focus on substantive words (length > 3) to avoid forcing generic grammar particles
            meaningful_words = [w for w in words if len(w) > 3]

            if meaningful_words:
                constraint_word = max(meaningful_words, key=len)
            elif words:
                constraint_word = max(words, key=len)
            else:
                constraint_word = ""

            if constraint_word:
                constraint_ids = tokenizer(constraint_word, add_special_tokens=False).input_ids
                if constraint_ids:
                    force_words = [constraint_ids]

        # 4. Prepare inputs
        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        # 5. Generate text
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("zul_Latn"),
                max_length=200,
                num_beams=5,                   # Constrained beam search requires num_beams >= 2
                force_words_ids=force_words,   # Evaluates to None if match was below threshold
                trust_remote_code=True
            )

        decoded = tokenizer.decode(generated[0], skip_special_tokens=True)

        if 'clean_translation' in globals():
            decoded = clean_translation(decoded)

        outputs.append(decoded)

    total_time = time.time() - start
    latency = total_time / len(texts)
    throughput = len(texts) / total_time

    return outputs, latency, throughput

In [ ]:
print("Running Constrained Beam Search...")
predictions_constrained, lat_c, thr_c = translate_with_lexical_constraints(
    sources[:10], # Testing on a small subset first
    retriever,
    index,
    reference_targets
)

print(predictions_constrained)

Running Constrained Beam Search...
['"Senawo manje amagundane anenyanga ezine ubudala angenawo isifo sikashukela futhi ake aba nesifo sikashukela," wanezela.', 'UDkt. Ehud Ur, onguprofesa wezokwelapha eDalhousie University eHalifax, eNova Scotia futhi ongusihlalo wendawo yokwelashwa kanye nesayensi ye-Canadian Diabetes Association waxwayisa ngokuthi lolu cwaningo lusesezinsukwini zalo zokuqala.', 'Njengezinye izazi, ungabaza ukuthi isifo sikashukela singalashwa yini, uphawula ukuthi lokhu okutholakele akunandaba kubantu asebenesifo sikashukela sohlobo 1.', 'NgoMsombuluko, uSara Danius, unobhala oqhubekayo weKomidi leNobel lezincwadi eSweden Academy, wamemezela esidlangalaleni phakathi nohlelo lwesiteshi somsakazo ku-Sveriges Radio eSweden ukuthi ikomidi, elingakhoni ukufinyelela ku-Bob Dylan ngqo mayelana nokuwina umklomelo ka-Nobel wezincwadi ka-2016, liyekile imizamo yalo yokufinyelela kuye.', 'U-Danius uthe, "Manje asenzi lutho. Ngishayele futhi ngathumela ama-imeyili kumuntu oseben

In [ ]:
cleaned_predictions = [clean_translation(pred) for pred in predictions_constrained]

In [ ]:
import sacrebleu

# Calculate the chrF++ score for the constrained predictions
chrf_constrained = sacrebleu.corpus_chrf(predictions_constrained, [references])

print("\n=== Evaluation Results ===")
# Assuming 'chrf.score' from baseline is still stored in memory
print(f"Baseline chrF++:             {chrf.score:.2f}")
print(f"Constrained Decoding chrF++: {chrf_constrained.score:.2f}")
print(f"Constrained Latency:         {lat_c:.4f} sec/sample")
print(f"Constrained Throughput:      {thr_c:.2f} samples/sec")


=== Evaluation Results ===
Baseline chrF++:             54.70
Constrained Decoding chrF++: 57.21
Constrained Latency:         1.0447 sec/sample
Constrained Throughput:      0.96 samples/sec


In [ ]:
results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf_constrained.score,
    "latency": lat_c,
    "throughput": thr_c,
    "num_samples": len(sources)
}

# Save metrics, ref, sources
with open("constrained_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open("constrained_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("constrained_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)

#### Post-Processing & Unicode Normalization

In [ ]:
import unicodedata
import re

def clean_translation_advanced(text: str) -> str:
    """
    Applies Unicode normalization and advanced regex heuristics to clean up
    common NMT subword tokenization and formatting artifacts.
    """
    if not isinstance(text, str):
        return ""

    # ---------------------------------------------------------
    # 1. Unicode Normalization
    # ---------------------------------------------------------
    text = unicodedata.normalize('NFC', text)

    # ---------------------------------------------------------
    # 2. Punctuation and Quote Formatting
    # ---------------------------------------------------------
    # Remove spaces mistakenly placed before terminal punctuation
    text = re.sub(r'\s+([?.!,:;])', r'\1', text)

    # Deduplicate repeating punctuation (common NMT stuttering artifact)
    text = re.sub(r'([?.!,:;])\1+', r'\1', text)

    # Fix spacing inside parentheses and brackets
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\[\s+', '[', text)
    text = re.sub(r'\s+\]', ']', text)

    # Fix basic quotation spacing
    text = re.sub(r'(["\'])\s+(.*?)\s+\1', r'\1\2\1', text)

    # ---------------------------------------------------------
    # 3. Subword & Hyphenation Artifacts (Crucial for Nguni languages)
    # ---------------------------------------------------------
    # Fix detached hyphens often left by BPE tokenizers (e.g., "oku - hle" -> "oku-hle")
    text = re.sub(r'\s+-\s+', '-', text)

    # Remove dangling hyphens attached to nothing at the start of a word
    text = re.sub(r'(?<=\s)-(?=\w)', '', text)

    # ---------------------------------------------------------
    # 4. Numbers and Currency
    # ---------------------------------------------------------
    # Fix separated currency symbols (e.g., "R 100" -> "R100", "$ 50" -> "$50")
    text = re.sub(r'([R$£€])\s+(\d)', r'\1\2', text)

    # ---------------------------------------------------------
    # 5. Final Whitespace & Casing Cleanup
    # ---------------------------------------------------------
    text = re.sub(r'\s{2,}', ' ', text)
    text = text.strip()

    # Ensure the sentence begins with a capital letter
    if text and text[0].islower():
        text = text[0].upper() + text[1:]

    return text

In [ ]:
import sacrebleu

print("Applying advanced post-processing to baseline predictions...")

# 1. Apply ONLY the advanced cleaner to the raw baseline predictions
# (Ensure 'predictions' is the output from your standard 'translate()' function)
cleaned_baseline_predictions = [clean_translation_advanced(pred) for pred in predictions]

# 2. Evaluate the cleaned output
chrf_post_processed = sacrebleu.corpus_chrf(cleaned_baseline_predictions, [references], word_order=2)

# 3. Compare the scores
print("\n=== Post-Processing Evaluation Results ===")
print(f"Baseline chrF++:               {chrf.score:.2f}")
print(f"Post-Processed chrF++:         {chrf_post_processed.score:.2f}")

# Latency and throughput remain identical to the baseline generation step
# plus a negligible microsecond overhead for the regex operations.
print(f"Post-Processed Latency:        {latency:.4f} sec/sample")
print(f"Post-Processed Throughput:     {throughput:.2f} samples/sec")

Applying advanced post-processing to baseline predictions...

=== Post-Processing Evaluation Results ===
Baseline chrF++:               54.70
Post-Processed chrF++:         49.13
Post-Processed Latency:        0.1766 sec/sample
Post-Processed Throughput:     5.66 samples/sec


In [ ]:
results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf_post_processed.score,
    "latency": latency,
    "throughput": throughput,
    "num_samples": len(sources)
}

# Save metrics
with open("post_processing_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open("post_processing_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("post_processing_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)